# Phase 2: Feature Engineering

## Objective

The objective of this phase is to create meaningful features that help the forecasting model learn sales patterns.

The engineered features include:

- Temporal features
- Holiday indicators
- Historical SKU statistics
- Target transformation
- Lag features
- Rolling statistics

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import holidays

import warnings
warnings.filterwarnings("ignore")

In [1]:
import sys
sys.path.append("../")

from src.utils.utils import (
    engineer_temporal_features,
    engineer_sku_features,
    cal_ma_lag_features
)

In [4]:
data_processed = pd.read_csv(
    "../data/processed/data_processed.csv",
    parse_dates=["shipped_date"]
)

data_processed.head()

,shipped_date,sku,channel,qty,revenue,COGS,MOQ_orders
0,2021-01-01,089A0E,ADS,190,5027.40,2926.00,56460
1,2021-01-01,089A0E,AWH,30,793.80,154.00,56460
2,2021-01-01,0FKFLA,AWH,780,32028.36,13104.00,427545
3,2021-01-01,0G8Z4M,AWH,85,1307.81,595.00,2516
4,2021-01-01,0NFJ14,FBM,38,2127.47,1723.25,18734


In [5]:
data_processed.info()
data_processed.shape

<class 'pandas.DataFrame'>
RangeIndex: 48345 entries, 0 to 48344
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   shipped_date  48345 non-null  datetime64[us]
 1   sku           48345 non-null  str           
 2   channel       48345 non-null  str           
 3   qty           48345 non-null  int64         
 4   revenue       48285 non-null  float64       
 5   COGS          47710 non-null  float64       
 6   MOQ_orders    48345 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(2), str(2)
memory usage: 2.6 MB


(48345, 7)

In [6]:
data_temp = engineer_temporal_features(data_processed)

data_temp.head()

,shipped_date,sku,channel,qty,revenue,COGS,MOQ_orders,month,day,dayofweek,weekofyear,quarter,is_weekend,is_even_day
0,2021-01-01,089A0E,ADS,190,5027.40,2926.00,56460,1,1,4,53,1,0,0
1,2021-01-01,089A0E,AWH,30,793.80,154.00,56460,1,1,4,53,1,0,0
2,2021-01-01,0FKFLA,AWH,780,32028.36,13104.00,427545,1,1,4,53,1,0,0
3,2021-01-01,0G8Z4M,AWH,85,1307.81,595.00,2516,1,1,4,53,1,0,0
4,2021-01-01,0NFJ14,FBM,38,2127.47,1723.25,18734,1,1,4,53,1,0,0


In [7]:
data_temp[
    [
        "month",
        "day",
        "dayofweek",
        "weekofyear",
        "quarter",
        "is_weekend",
        "is_even_day"
    ]
].describe()

,month,day,dayofweek,weekofyear,quarter,is_weekend,is_even_day
count,48345.000000,48345.00000,48345.000000,48345.000000,48345.000000,48345.000000,48345.000000
mean,6.682759,15.62567,3.199359,27.600103,2.558858,0.309898,0.480753
std,3.493408,8.90205,1.994908,15.192703,1.128354,0.462456,0.499635
min,1.000000,1.00000,0.000000,1.000000,1.000000,0.000000,0.000000
25%,4.000000,8.00000,1.000000,14.000000,2.000000,0.000000,0.000000
50%,7.000000,16.00000,3.000000,28.000000,3.000000,0.000000,0.000000
75%,10.000000,23.00000,5.000000,41.000000,4.000000,1.000000,1.000000
max,12.000000,31.00000,6.000000,53.000000,4.000000,1.000000,1.000000


## Holiday Feature

In [8]:
holiday_list = [

    "2021-01-01",
    "2021-01-02",
    "2021-01-03",

    "2021-02-10",
    "2021-02-11",
    "2021-02-12",
    "2021-02-13",
    "2021-02-14",
    "2021-02-15",
    "2021-02-16",

    "2021-04-21",

    "2021-04-30",
    "2021-05-01",
    "2021-05-02",
    "2021-05-03",

    "2021-09-02",
    "2021-09-03",
    "2021-09-04",
    "2021-09-05",

    "2021-12-24",
    "2021-12-25",
]

holiday_list = pd.to_datetime(holiday_list)

data_temp["is_holiday"] = (
    data_temp["shipped_date"]
    .isin(holiday_list)
    .astype(int)
)

In [9]:
data_temp["is_holiday"].value_counts()

is_holiday
0    45217
1     3128
Name: count, dtype: int64

## SKU Historical Features

In [10]:
data_feature = engineer_sku_features(data_temp)

data_feature.head()

,shipped_date,sku,channel,qty,revenue,COGS,MOQ_orders,month,day,dayofweek,weekofyear,quarter,is_weekend,is_even_day,is_holiday,dow,mean_qty_sku_dow,mean_qty_sku_month
335,2021-01-03,017SAD,ADS,75,3358.95,1660.05,1425,1,3,6,53,1,1,0,1,6,0.0,0.0
1643,2021-01-15,017SAD,ADS,75,3358.95,1660.05,1425,1,15,4,2,1,0,0,0,4,75.0,75.0
5716,2021-02-16,017SAD,ADS,75,3358.95,1660.05,1425,2,16,1,7,1,0,1,1,1,75.0,75.0
5717,2021-02-16,017SAD,LAL,75,2310.00,2009.70,1425,2,16,1,7,1,0,1,1,1,75.0,75.0
6869,2021-02-26,017SAD,ADS,75,3358.95,1660.05,1425,2,26,4,8,1,0,1,0,4,75.0,75.0


In [11]:
data_feature[
    [
        "mean_qty_sku_dow",
        "mean_qty_sku_month"
    ]
].describe()

,mean_qty_sku_dow,mean_qty_sku_month
count,48345.000000,48345.000000
mean,202.936436,209.975410
std,244.540629,257.161437
min,0.000000,0.000000
25%,43.700000,46.142857
50%,96.000000,100.000000
75%,257.500000,269.500000
max,1960.000000,2002.000000


## Log Transformation

In [12]:
data_feature["qty_log"] = np.log1p(data_feature["qty"])

In [13]:
data_feature["qty_log"].describe()

count    48345.000000
mean         4.528390
std          1.265715
min          0.000000
25%          3.555348
50%          4.394449
75%          5.476464
max          7.602401
Name: qty_log, dtype: float64

## Lag & Rolling Features

In [14]:
data_final = cal_ma_lag_features(data_feature)

data_final.head()

,shipped_date,sku,channel,qty,revenue,COGS,MOQ_orders,month,day,dayofweek,...,mean_qty_sku_dow,mean_qty_sku_month,qty_log,lag1,lag7,lag30,rolling_mean_7,rolling_mean_30,rolling_std_7,rolling_std_30
335,2021-01-03,017SAD,ADS,75,3358.95,1660.05,1425,1,3,6,...,0.0,0.0,4.330733,NaN,NaN,NaN,75.0,75.0,NaN,NaN
1643,2021-01-15,017SAD,ADS,75,3358.95,1660.05,1425,1,15,4,...,75.0,75.0,4.330733,75.0,NaN,NaN,75.0,75.0,0.0,0.0
5716,2021-02-16,017SAD,ADS,75,3358.95,1660.05,1425,2,16,1,...,75.0,75.0,4.330733,75.0,NaN,NaN,75.0,75.0,0.0,0.0
5717,2021-02-16,017SAD,LAL,75,2310.00,2009.70,1425,2,16,1,...,75.0,75.0,4.330733,75.0,NaN,NaN,75.0,75.0,0.0,0.0
6869,2021-02-26,017SAD,ADS,75,3358.95,1660.05,1425,2,26,4,...,75.0,75.0,4.330733,75.0,NaN,NaN,75.0,75.0,0.0,0.0


In [15]:
data_final.isnull().sum()

shipped_date              0
sku                       0
channel                   0
qty                       0
revenue                   0
COGS                      0
MOQ_orders                0
month                     0
day                       0
dayofweek                 0
weekofyear                0
quarter                   0
is_weekend                0
is_even_day               0
is_holiday                0
dow                       0
mean_qty_sku_dow          0
mean_qty_sku_month        0
qty_log                   0
lag1                    676
lag7                   3712
lag30                 12561
rolling_mean_7            0
rolling_mean_30           0
rolling_std_7           676
rolling_std_30          676
dtype: int64

In [16]:
data_final.to_csv(
    "../data/processed/data_engineered.csv",
    index=False
)

print("Feature engineering completed successfully.")

Feature engineering completed successfully.


In [17]:
check = pd.read_csv("../data/processed/data_engineered.csv")

check.head()

,shipped_date,sku,channel,qty,revenue,COGS,MOQ_orders,month,day,dayofweek,...,mean_qty_sku_dow,mean_qty_sku_month,qty_log,lag1,lag7,lag30,rolling_mean_7,rolling_mean_30,rolling_std_7,rolling_std_30
0,2021-01-03,017SAD,ADS,75,3358.95,1660.05,1425,1,3,6,...,0.0,0.0,4.330733,NaN,NaN,NaN,75.0,75.0,NaN,NaN
1,2021-01-15,017SAD,ADS,75,3358.95,1660.05,1425,1,15,4,...,75.0,75.0,4.330733,75.0,NaN,NaN,75.0,75.0,0.0,0.0
2,2021-02-16,017SAD,ADS,75,3358.95,1660.05,1425,2,16,1,...,75.0,75.0,4.330733,75.0,NaN,NaN,75.0,75.0,0.0,0.0
3,2021-02-16,017SAD,LAL,75,2310.00,2009.70,1425,2,16,1,...,75.0,75.0,4.330733,75.0,NaN,NaN,75.0,75.0,0.0,0.0
4,2021-02-26,017SAD,ADS,75,3358.95,1660.05,1425,2,26,4,...,75.0,75.0,4.330733,75.0,NaN,NaN,75.0,75.0,0.0,0.0


In [18]:
check.shape

(48345, 26)